In [41]:
import Gmsh: gmsh
import Pkg; Pkg.add("Distributions")
using Gridap, GridapGmsh
using Random
using Distributions

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [42]:
function create_square_model(h)
    gmsh.initialize()
    gmsh.model.add("unit_square");
    gmsh.model.geo.addPoint(0.0, 0.0, 0.0, h, 1); # last argument is optional identifier, unique per dimension
    gmsh.model.geo.addPoint(1.0, 0.0, 0.0, h, 2);
    gmsh.model.geo.addPoint(1.0, 1.0, 0.0, h, 3);
    gmsh.model.geo.addPoint(0.0, 1.0, 0.0, h, 4);
    gmsh.model.geo.addLine(1, 2, 1); 
    gmsh.model.geo.addLine(2, 3, 2); # line 2 goes from point 2 to point 3
    gmsh.model.geo.addLine(3, 4, 3);
    gmsh.model.geo.addLine(4, 1, 4);
    
    gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1);

    gmsh.model.geo.addPlaneSurface([1], 1);

    gmsh.model.geo.synchronize();

    # Define physical groups without the string argument
    edges_tag = gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4])   # edges
    corners_tag = gmsh.model.addPhysicalGroup(0, [1, 2, 3, 4]) # corners
    domain_tag = gmsh.model.addPhysicalGroup(2, [1])           # surface
    
    # Set names for the physical groups
    gmsh.model.setPhysicalName(1, edges_tag, "boundary")
    gmsh.model.setPhysicalName(0, corners_tag, "boundary")
    gmsh.model.setPhysicalName(2, domain_tag, "domain")

    gmsh.model.mesh.generate(2); # dimension is 2
    
    model = GmshDiscreteModel(gmsh);
    gmsh.finalize();
    return model
end


create_square_model (generic function with 1 method)

In [43]:
h = 0.04; # target mesh size
model = create_square_model(h) # fix above function above using the tutorial
order = 1
reffe = ReferenceFE(lagrangian, Float64, order)
V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
U = TrialFESpace(V, 0.0)


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.00016423s, CPU 0.000156s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.012325s, CPU 0.011583s)
Info    : 790 nodes 1582 elements


TrialFESpace()

In [44]:
Ω = Triangulation(model)
dΩ = Measure(Ω, 2 * order)

GenericMeasure()

In [45]:
n=7
Δt, T = 2.0^(-n), 2*π/3 # time step, final time
num_steps = Int(round(T / Δt))

268

In [46]:
#g(x) = x[1]*(1-x[1]) *x[2]*(1-x[2])
#Δg(x) = -2*x[2]*(1 - x[2]) - 2*x[1]*(1 - x[1])

#f(x) = -9*g(x) - Δg(x)
f(x) = x[1]*(1-x[1]) *x[2]*(1-x[2])
fx = x->f(x) 

#37 (generic function with 1 method)

In [47]:
u₀ = x -> 0.0
v₀ = x -> 0.0

#41 (generic function with 1 method)

In [48]:
m(u, v) = ∫(u*v)dΩ
k(u, v) = ∫(∇(u) ⋅ ∇(v))dΩ

M = assemble_matrix(m, U, V)
K = assemble_matrix(k, U, V)

LHSpos = (2/Δt)*M + (Δt/2)*K
RHSpos = (2/Δt)*M - (Δt/2)*K

690×690 SparseArrays.SparseMatrixCSC{Float64, Int64} with 4632 stored entries:
⎡⠑⢄⢈⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡸⡞⣈⢔⣺⢤⢖⡀⠀⣢⢮⢊⠡⠈⡺⣷⣷⣴⡶⎤
⎢⠂⠐⠱⣦⡀⠀⠐⠀⠀⠀⠄⠐⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠹⣦⢅⠅⡄⠰⢀⡸⠀⠁⠸⠁⠘⠉⠉⠈⠉⠁⎥
⎢⠀⠀⠀⠈⠻⣦⣄⡅⠀⢀⠀⠀⠃⠆⠀⡄⠀⠀⠀⢀⠀⠀⠀⠀⢌⠀⣁⠈⣧⡙⣢⠀⠁⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠐⠀⠄⠽⠟⣥⡄⠘⠼⠶⡂⠂⠀⠔⠀⠀⠀⠀⠀⠀⠀⠀⠁⠀⠵⢈⠭⢼⣃⠐⠃⠃⠧⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⢀⣀⠉⣻⣾⣡⣅⠀⠀⠀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⡫⣠⢋⣹⣦⣂⡰⡒⠂⠀⠀⠀⠀⎥
⎢⠀⠀⢀⠁⠀⠀⢲⡇⠅⢾⣿⣿⣖⣧⡢⢂⠀⠀⠀⠨⠀⠀⠀⠉⠐⠀⣴⠬⠵⢗⠆⣟⡨⠅⢐⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠩⠄⠨⠈⠀⠀⠼⣽⣻⣾⢬⠺⣡⢌⠠⠀⢱⣀⡀⠈⠀⠐⣝⡡⠤⢽⡭⠒⠏⠪⠰⠀⠐⠀⠀⠂⎥
⎢⠀⠀⠀⠀⠀⠤⢀⠄⠀⠠⠨⢊⣢⡓⣱⢞⠯⢘⠧⠠⣀⢠⢅⠙⠀⢀⠋⠢⢾⢤⠦⢍⣔⣊⣤⡔⢀⠀⡂⡀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡁⢞⣋⢃⣻⣾⡂⠀⢠⢼⢋⣈⡘⠔⡉⢙⠀⡓⡖⣀⣢⢂⣄⠉⡗⢮⠀⠆⎥
⎢⠀⠀⠀⠀⠀⢀⠀⠀⠀⠀⡀⡀⠀⠂⠉⡃⠈⠈⠻⣦⡀⡄⢀⠸⠐⠁⠃⠐⣘⠈⡘⡎⢚⣚⣳⡄⠁⠊⠄⠀⎥
⎢⣀⡠⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢲⠀⣘⣀⣖⠀⠬⢻⢖⡉⢀⡡⠴⣰⠔⠂⠀⠥⡕⠁⠋⠏⠣⣾⣲⡦⡢⎥
⎢⡚⢩⠳⣦⠀⠀⠀⠀⠀⠀⡄⠀⡀⠈⣅⠑⡋⢰⣀⡐⠃⢈⢱⢖⢧⡘⡉⢐⢀⠎⠀⢠⠳⠮⠶⢖⣁⢈⡉⡈⎥
⎢⣰⣱⠅⠕⠂⠑⠁⠀⠀⠀⠐⠀⢀⠀⠀⢀⢒⠌⠔⠀⢁⡎⣉⠳⠕⢅⠂⠘⠊⠃⢀⠒⡪⠓⠀⢺⢄⡠⡿⡤⎥
⎢⢠⢗⢀⡉⡁⠘⡑⢃⡤⡢⡐⡟⠗⡹⠫⡀⣇⢈⢉⠀⢐⠞⢃⢈⣈⠀⠱⣦⢸⡂⡠⢐⡤⣐⠁⡠⡶⣪⠡⡢⎥
⎢⠀⠈⣀⡰⣍⠻⣃⣇⡤⢚⢵⢇⣄⣇⠚⣗⢤⠠⡒⠘⠈⠀⡠⠔⠮⠀⠲⠲⡵⣯⠝⠄⣭⡵⡨⠄⠁⢀⢁⠀⎥
⎢⡨⣞⠄⠀⠈⠚⢉⠘⠳⣾⣬⢥⢣⠋⡌⢇⠘⢩⡲⠬⢅⠧⠀⣀⢠⠐⢀⢊⠓⠅⠛⣤⡻⡽⠧⠧⡐⣠⠩⡄⎥
⎢⠎⡐⠖⠂⠁⠀⠭⠀⢈⡸⠆⠎⡫⡁⡰⢹⠨⢚⣺⢰⡥⠀⡹⡆⢮⠊⢀⢫⢇⡿⣟⡮⣿⣿⣗⠃⠈⠤⠀⠀⎥
⎢⣢⡠⡖⠀⠀⠀⠉⠃⠸⠈⠐⠐⠐⠂⢀⠿⡄⠙⠙⠾⠯⡁⢸⢇⣠⣀⠁⡠⠂⠎⠭⡇⠽⠙⢿⢗⠐⢈⡀⢈⎥
⎢⢽⣿⡃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠐⠀⠀⠐⡹⣍⡡⠀⢺⣻⡁⢘⠀⡱⡸⣫⠁⢀⠐⣨⠂⡄⡐⢀⡛⢌⠋⢟⎥
⎣⢰⡿⠇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⠀⠈⠨⠠⠄⠀⠁⠨⡫⡃⠨⠛⡯⠡⡢⠁⠐⠃⠦⠀⠀⡀⢈⣯⢄⢕⢕⎦

In [49]:
uh0 = interpolate_everywhere(u₀, U)   # U^0
vh0 = interpolate_everywhere(v₀, U)   # V^0

SingleFieldFEFunction():
 num_cells: 1478
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 9552038962578635320

In [50]:
if !isdir("tmp") mkdir("tmp") end

createpvd("wave") do pvd
    pvd[0.0] = createvtk(Ω, "tmp/stoc_wavev2_0.vtu",
                         cellfields=["uh" => uh0, "vh" => vh0])

    # dof-vectors
    uvec = get_free_dof_values(uh0)  # U^0
    vvec = get_free_dof_values(vh0)  # V^0

    for n in 1:(num_steps-1)
        t_n = n*Δt
        Δβ = rand(Normal(0,sqrt(Δt)))
        b(v) = ∫(fx*v)dΩ
        Fn = assemble_vector(b, V)
        noise =  Δβ * Fn


        #Solve for U^n
        rhs_u = RHSpos * uvec + (2*M)*vvec + 2*noise
        uvec_new = LHSpos \ rhs_u

        #update V^n
        rhs_v = M*vvec - (Δt/2)*K*(uvec_new + uvec) + noise
        #rhs_v = ((2/Δt) * M) * (uvec_new - uvec) - M*vvec -noise
        vvec_new = M \ rhs_v

        uh = FEFunction(U, uvec_new)
        vh = FEFunction(U, vvec_new)
        
        #save
        vtufilename = "tmp/stoc_wavev2_$(n).vtu"
        pvd[t_n] = createvtk(Ω, vtufilename, cellfields=["uh" => uh, "vh" => vh])

        # update
        uvec = uvec_new
        vvec = vvec_new
    end
end

269-element Vector{String}:
 "wave.pvd"
 "tmp/stoc_wavev2_0.vtu"
 "tmp/stoc_wavev2_1.vtu"
 "tmp/stoc_wavev2_2.vtu"
 "tmp/stoc_wavev2_3.vtu"
 "tmp/stoc_wavev2_4.vtu"
 "tmp/stoc_wavev2_5.vtu"
 "tmp/stoc_wavev2_6.vtu"
 "tmp/stoc_wavev2_7.vtu"
 "tmp/stoc_wavev2_8.vtu"
 "tmp/stoc_wavev2_9.vtu"
 "tmp/stoc_wavev2_10.vtu"
 "tmp/stoc_wavev2_11.vtu"
 ⋮
 "tmp/stoc_wavev2_256.vtu"
 "tmp/stoc_wavev2_257.vtu"
 "tmp/stoc_wavev2_258.vtu"
 "tmp/stoc_wavev2_259.vtu"
 "tmp/stoc_wavev2_260.vtu"
 "tmp/stoc_wavev2_261.vtu"
 "tmp/stoc_wavev2_262.vtu"
 "tmp/stoc_wavev2_263.vtu"
 "tmp/stoc_wavev2_264.vtu"
 "tmp/stoc_wavev2_265.vtu"
 "tmp/stoc_wavev2_266.vtu"
 "tmp/stoc_wavev2_267.vtu"